# White Hat / Black Hat

## 1. Setup

In [ ]:
from pathlib import Path


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root()
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)


def save_figure(fig, name: str) -> None:
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
BLUE   = "#2176AE"
ORANGE = "#E76F51"
TEAL   = "#2A9D8F"
SLATE  = "#264653"
RED    = "#E63946"
GRAY   = "#C8C8C8"

plt.rcParams.update({
    "figure.dpi":          130,
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           False,
    "font.family":         "sans-serif",
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.labelsize":      10,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "legend.fontsize":     9,
    "legend.frameon":      False,
    "figure.titlesize":    13,
    "figure.titleweight":  "bold",
})

## 2. Data Loading

In [ ]:
df_state = pd.read_csv(CLEAN_DIR / "emission_state_clean.csv", parse_dates=["DATE"])
df_net = pd.read_csv(CLEAN_DIR / "emission_network_clean.csv", parse_dates=["DATE"])

CODE2NAME = {
    "AL": "Albania",     "AM": "Armenia",     "AT": "Austria",      "AZ": "Azerbaijan",
    "BA": "Bosnia",      "BE": "Belgium",      "BG": "Bulgaria",     "CH": "Switzerland",
    "CY": "Cyprus",      "CZ": "Czechia",      "DE": "Germany",      "DK": "Denmark",
    "EE": "Estonia",     "ES": "Spain",        "FI": "Finland",      "FR": "France",
    "GB": "UK",          "GE": "Georgia",      "GR": "Greece",       "HR": "Croatia",
    "HU": "Hungary",     "IE": "Ireland",      "IL": "Israel",       "IS": "Iceland",
    "IT": "Italy",       "LT": "Lithuania",    "LU": "Luxembourg",   "LV": "Latvia",
    "MA": "Morocco",     "MD": "Moldova",      "ME": "Montenegro",   "MK": "N.Macedonia",
    "MT": "Malta",       "NL": "Netherlands",  "NO": "Norway",       "PL": "Poland",
    "PT - Lisbon FIR":      "Portugal",
    "PT - Santa Maria FIR": "PT Azores",
    "RO": "Romania",     "RS": "Serbia",       "SE": "Sweden",       "SI": "Slovenia",
    "SK": "Slovakia",    "TR": "Turkey",       "UA": "Ukraine",
}
print(f"df_state: {len(df_state):,} rows  |  df_net: {len(df_net):,} rows")


## 3. White Hat: State-Level CO₂ Totals

In [ ]:
state_tot = (
    df_state.groupby("AREA")["CO2_KG"].sum()
    .reset_index()
    .sort_values("CO2_KG", ascending=False)
    .head(25)
)
state_tot["name"] = state_tot["AREA"].map(CODE2NAME).fillna(state_tot["AREA"])

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(state_tot["name"], state_tot["CO2_KG"] / 1e9,
               color=SLATE, edgecolor="white")
ax.invert_yaxis()
ax.set_xlabel("Total CO₂ 2019–2024 (Mt)")
ax.set_title("Top 25 European states by aviation CO₂ (2019–2024)")

for bar, val in zip(bars, state_tot["CO2_KG"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val / 1e9:.1f}", va="center", fontsize=8, color="#444")

plt.tight_layout()
save_figure(fig, "whitehat_a")
plt.show()

### Black Hat 

**Manipulation:** broken x-axis with no tick labels. The three panels share the y-axis but use incompatible x-scales, making Turkey and the UK appear proportionally similar to mid-tier emitters. The missing axis values prevent any honest magnitude comparison.

In [ ]:
state_bh = state_tot.iloc[::-1].reset_index(drop=True)
names    = state_bh["name"].values
vals     = state_bh["CO2_KG"].values / 1e9

fig, (ax1, ax2, ax3) = plt.subplots(
    1, 3, sharey=True, figsize=(13, 8),
    gridspec_kw={"width_ratios": [4, 0.5, 0.6], "wspace": 0.02},
)
for ax in [ax1, ax2, ax3]:
    ax.barh(names, vals, color=SLATE, edgecolor="white", height=0.75)

ax1.set_xlim(0, 22)
ax2.set_xlim(25, 35)
ax3.set_xlim(78, 122)

ax1.spines["right"].set_visible(False)
ax1.spines["top"].set_visible(False)
ax2.spines["left"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.spines["top"].set_visible(False)
ax2.tick_params(left=False)
ax3.spines["left"].set_visible(False)
ax3.spines["top"].set_visible(False)
ax3.tick_params(left=False)

ax1.set_xticks([])
ax2.set_xticks([])
ax3.set_xticks([])

fig.text(0.5, 0.02, "Total CO₂ 2019–2024 (Mt)", ha="center", fontsize=10)
fig.suptitle("Top states by aviation CO₂ emissions", fontsize=13, y=0.97)

plt.tight_layout(rect=[0, 0.04, 1, 0.95])
save_figure(fig, "blackhat_a")
plt.show()